# 🔥 Análise de Queimadas no Brasil com Dados de Satélite da NASA

Projeto de Data Science desenvolvido por **Gabriel Cardoso** durante a graduação em Engenharia de Software na FIAP.

A análise explora focos de calor detectados por satélites da NASA no Brasil entre **25/05/2025 e 25/05/2026**, utilizando estatística descritiva, análise exploratória de dados e visualizações em Python.

> O CSV original não é versionado neste repositório devido ao tamanho do arquivo. Consulte `data/README.md` para instruções sobre os dados.

## ▶️ Como executar

1. Obtenha o arquivo conforme descrito em `data/README.md`.
2. Salve-o em `data/fire_archive_SV-C2_755280.csv`.
3. Execute as células deste notebook em sequência.


## Importação das bibliotecas


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

Leitura do arquivo CSV

In [ ]:
from pathlib import Path

caminhos_possiveis = [
    Path('../data/fire_archive_SV-C2_755280.csv'),
    Path('data/fire_archive_SV-C2_755280.csv')
]

data_path = next((path for path in caminhos_possiveis if path.exists()), None)

if data_path is None:
    raise FileNotFoundError('Arquivo CSV não encontrado. Consulte data/README.md.')

df = pd.read_csv(data_path)


Visualizar os primeiros dados

In [ ]:
df.head()

## Dicionário de dados

| Coluna | Significado |
|----------|----------|
| latitude | Coordenada geográfica que indica a posição do foco de incêndio no sentido norte-sul do território brasileiro. |
| longitude | Coordenada geográfica que indica a posição do foco de incêndio no sentido leste-oeste, permitindo localizar a ocorrência no mapa. |
| brightness | Brilho térmico detectado pelo sensor do satélite. Valores mais altos geralmente indicam temperaturas mais elevadas e maior atividade de combustão. |
| scan | Dimensão da área observada pelo sensor na direção da varredura do satélite. Auxilia na estimativa da área monitorada. |
| track | Dimensão da área observada pelo sensor na direção do deslocamento orbital do satélite. Complementa a definição da área detectada. |
| acq_date | Data em que o foco de incêndio foi registrado pelo satélite. |
| acq_time | Horário em que o foco foi detectado pelo sistema de monitoramento. |
| satellite | Satélite responsável pela captura das informações utilizadas na análise. |
| instrument | Equipamento sensor embarcado no satélite responsável pela detecção dos focos de calor. |
| confidence | Nível de confiança atribuído pelo sistema à detecção do foco de incêndio. Quanto maior a confiança, maior a probabilidade de a ocorrência ser real. |
| version | Versão do algoritmo de processamento utilizado pela NASA para gerar os dados. |
| bright_t31 | Temperatura registrada em uma faixa específica do infravermelho térmico, utilizada para auxiliar na identificação e validação dos focos de calor. |
| frp | Fire Radiative Power (Potência Radiativa do Fogo). Mede a energia liberada pela queimada e é um dos principais indicadores da intensidade do incêndio. |
| daynight | Indica se a detecção ocorreu durante o dia (D) ou durante a noite (N). |
| type | Classificação do evento detectado pelo sistema, diferenciando tipos de ocorrências relacionadas ao foco de calor. |

Inspecionar estrutura da base

In [ ]:
df.info()

## Primeira leitura da base

A base possui **853.047 registros** de focos detectados no Brasil entre 25/05/2025 e 25/05/2026, distribuídos em 15 colunas.

Antes de partir para os cálculos, observei principalmente três grupos de informações: localização (`latitude` e `longitude`), momento da detecção (`acq_date`, `acq_time` e `daynight`) e intensidade térmica (`brightness`, `bright_t31` e `frp`).

Também conferi a presença de valores ausentes e a estrutura dos tipos de dados para entender se seria necessário algum tratamento antes da análise.


Ver estatísticas iniciais

In [ ]:
df.describe()

## O que apareceu nas estatísticas iniciais

O primeiro ponto que me chamou atenção foi o FRP. A média ficou em aproximadamente **10,86 MW**, mas o valor máximo chegou a **2.074,84 MW**. Essa diferença já indicava que existiam registros muito acima do comportamento mais comum da base.

Os quartis reforçaram essa impressão: **75% dos registros possuem FRP abaixo de 10,72 MW**. Por isso, decidi olhar com mais cuidado para a dispersão e para os possíveis valores extremos nas etapas seguintes.


Verificar valores ausentes

In [ ]:
df.isnull().sum()

A verificação de valores ausentes mostrou que a base possui dados completos nas principais colunas analisadas, o que facilita a aplicação das técnicas estatísticas.

Foi realizada a remoção de possíveis registros duplicados para garantir maior consistência na análise.

In [ ]:
df = df.drop_duplicates()
df.shape

Padronizar nomes das colunas

In [ ]:
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
df.columns

## Importância das variáveis analisadas

Entre todas as variáveis disponíveis na base, destacam-se brightness, bright_t31 e FRP, pois estão diretamente relacionadas à intensidade dos focos de incêndio. O brilho térmico (brightness) representa a energia térmica detectada pelo satélite, enquanto a temperatura infravermelha (bright_t31) auxilia na confirmação da presença do foco de calor. Já o FRP é considerado um dos indicadores mais importantes, pois mede a quantidade de energia liberada pela queimada, permitindo avaliar sua intensidade e potencial impacto ambiental.

As variáveis latitude e longitude também possuem grande relevância, pois permitem identificar a distribuição geográfica dos incêndios ao longo do território brasileiro, contribuindo para análises espaciais e estratégias de monitoramento ambiental.

Selecionar colunas principais

In [ ]:
colunas_numericas = ['brightness', 'bright_t31', 'frp', 'scan', 'track']

df_analise = df[colunas_numericas]
df_analise.head()

As medidas de tendência central permitem identificar os valores típicos das variáveis analisadas, como brilho térmico, temperatura e potência radiativa do fogo.

In [ ]:
print("Média:")
display(df_analise.mean())

print("Mediana:")
display(df_analise.median())

print("Moda:")
display(df_analise.mode().iloc[0])

## Interpretação das Medidas de Tendência Central

A média do brilho térmico (brightness) foi de 334,67, indicando que a maior parte dos focos de incêndio apresentou níveis elevados de energia térmica.

A média da potência radiativa do fogo (FRP) foi de 10,86. Entretanto, a mediana foi de apenas 5,52, sugerindo que existem incêndios extremamente intensos que aumentam a média geral dos dados.

A moda do FRP foi 1,18, indicando que esse foi o valor mais frequente entre os registros analisados.

As medidas de dispersão indicam o quanto os dados variam em relação aos valores médios. Variáveis com maior coeficiente de variação apresentam maior instabilidade e dispersão.

In [ ]:
print("Amplitude:")
display(df_analise.max() - df_analise.min())

print("Variância:")
display(df_analise.var())

print("Desvio padrão:")
display(df_analise.std())

print("Coeficiente de variação (%):")
display((df_analise.std() / df_analise.mean()) * 100)

## Dispersão dos dados

Ao comparar as variáveis, o FRP foi o que apresentou a maior variação. O desvio padrão ficou em aproximadamente **21,06 MW**, acima da própria média de **10,86 MW**, e o coeficiente de variação chegou a cerca de **193,84%**.

Para mim, esse foi um dos resultados mais importantes da análise: os registros de FRP não ficam concentrados próximos da média. Existem poucos valores muito altos que alteram bastante a distribuição.

Já `brightness` e `bright_t31` apresentaram coeficientes de variação bem menores, de aproximadamente **5,36%** e **3,34%**, respectivamente.


Medidas posicionais

In [ ]:
df_analise.quantile([0.25, 0.50, 0.75])
df_analise.quantile([0.10, 0.25, 0.50, 0.75, 0.90])

## Quartis e percentis

Quando organizei o FRP por percentis, ficou mais fácil enxergar a concentração dos dados:

- 25% dos registros estão abaixo de **2,64 MW**;
- a mediana é **5,52 MW**;
- 75% estão abaixo de **10,72 MW**;
- 90% estão abaixo de **22,29 MW**.

Comparando esses valores com o máximo de **2.074,84 MW**, fica claro como uma parcela pequena dos registros está muito distante do restante da distribuição.


As medidas posicionais permitem observar como os dados estão distribuídos em diferentes partes da amostra, destacando quartis e percentis.

Análise de outliers com IQR

In [ ]:
Q1 = df['frp'].quantile(0.25)
Q3 = df['frp'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_frp = df[(df['frp'] < limite_inferior) | (df['frp'] > limite_superior)]

print("Limite inferior:", limite_inferior)
print("Limite superior:", limite_superior)
print("Quantidade de outliers:", len(outliers_frp))

A análise de outliers foi realizada pelo método IQR. Os valores extremos encontrados na variável FRP indicam focos de incêndio com potência radiativa muito acima do padrão observado na base.

#ANÁLISE DE DISTRIBUIÇÃO

Interpretar distribuição

A análise de distribuição mostra a concentração, dispersão e possíveis assimetrias dos dados. A variável FRP tende a apresentar grande variação, indicando que a maioria dos focos possui baixa ou média intensidade, enquanto poucos registros apresentam valores muito elevados.

In [ ]:
df_analise.describe()

Histograma

In [ ]:
plt.figure(figsize=(10,6))
sns.histplot(df['frp'], bins=50, kde=True)
plt.title('Distribuição da Potência Radiativa do Fogo - FRP')
plt.xlabel('FRP')
plt.ylabel('Frequência')
plt.show()

O histograma mostra que a maior parte dos valores de FRP encontra-se concentrada em níveis mais baixos, enquanto poucos registros apresentam valores muito elevados. Isso indica uma distribuição assimétrica, com presença de valores extremos.

Boxplot

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(x=df['frp'])
plt.title('Boxplot da Potência Radiativa do Fogo - FRP')
plt.xlabel('FRP')
plt.show()

## Leitura do boxplot

O boxplot tornou visual aquilo que as medidas anteriores já sugeriam. A maior parte dos registros de FRP está concentrada em valores baixos, enquanto aparecem diversos pontos bem acima da região central.

Pelo critério de IQR, esses pontos são classificados como outliers. Neste contexto, porém, não tratei automaticamente esses registros como erros: eles podem representar focos com potência radiativa muito acima do padrão da base e, por isso, são relevantes para a análise.


Gráfico de barras

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='daynight', data=df)
plt.title('Quantidade de Focos de Incêndio por Período')
plt.xlabel('Período')
plt.ylabel('Quantidade')
plt.show()

O gráfico de barras mostra a distribuição da quantidade de focos registrados entre os períodos diurno e noturno, permitindo comparar a frequência de ocorrências em cada situação.

Scatterplot

In [ ]:
plt.figure(figsize=(10,6))
sns.scatterplot(x='brightness', y='frp', data=df, alpha=0.3)
plt.title('Relação entre Brilho Térmico e FRP')
plt.xlabel('Brightness')
plt.ylabel('FRP')
plt.show()

O scatterplot indica uma tendência de crescimento conjunto entre brightness e FRP, sugerindo uma relação positiva entre brilho térmico e intensidade do fogo.

Heatmap de correlação

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df_analise.corr(), annot=True, cmap='coolwarm')
plt.title('Heatmap de Correlação entre Variáveis Numéricas')
plt.show()

## O que observei no heatmap

A maior correlação entre as variáveis selecionadas apareceu entre `brightness` e `bright_t31`, com aproximadamente **0,58**.

O FRP apresentou correlação positiva de cerca de **0,32** com `brightness` e **0,37** com `bright_t31`. Isso mostra uma associação linear positiva, mas não forte. Por isso, não faria sentido usar apenas uma dessas variáveis para explicar sozinha o comportamento do FRP.

Também apareceu uma correlação de aproximadamente **0,49** entre `scan` e `track`, variáveis ligadas às dimensões da área observada pelo sensor.


## Perguntas que guiaram a análise

### 1. Qual variável apresentou maior variabilidade?

O FRP foi a variável que mais se destacou. Seu coeficiente de variação ficou em aproximadamente **193,84%**, bem acima do observado em `brightness` e `bright_t31`.

### 2. Existem valores extremos relevantes?

Sim. O método IQR identificou diversos registros de FRP acima do limite superior. Mantive esses valores na interpretação porque, nesse contexto, eles podem representar focos muito mais intensos que o padrão da base.

### 3. Existe relação entre brilho térmico e FRP?

Existe uma associação positiva. A correlação entre `brightness` e FRP ficou em aproximadamente **0,32**. É uma relação perceptível, mas não forte.

### 4. O que dá para dizer sobre registros diurnos e noturnos?

A base permite comparar a quantidade de detecções classificadas como diurnas e noturnas. Essa comparação descreve os registros do período analisado, mas, sozinha, não permite concluir que incêndios ocorram mais em determinado período do dia.

### 5. E a distribuição geográfica?

A base possui latitude e longitude para cada detecção, então existe espaço para aprofundar a análise geográfica. Nesta versão do projeto, porém, meu foco principal foi a análise estatística das variáveis térmicas; por isso, não interpretei diferenças regionais sem construir uma análise espacial específica.


## Conclusão

Depois de explorar a base, o principal resultado para mim foi perceber o quanto o **FRP é assimétrico**. A mediana ficou em **5,52 MW**, enquanto a média foi de **10,86 MW** e o máximo passou de **2.000 MW**. Ou seja, poucos registros muito intensos têm bastante influência sobre a distribuição.

O método IQR e o boxplot ajudaram a confirmar essa característica. Em vez de simplesmente remover os valores extremos, preferi mantê-los na leitura porque eles fazem parte do fenômeno que estou analisando.

Nas correlações, `brightness` e `bright_t31` apresentaram a relação mais alta entre as variáveis selecionadas (**0,58**). O FRP também apresentou associação positiva com as duas medidas térmicas, mas sem uma correlação forte, o que reforça que essas relações precisam ser interpretadas com cuidado.

Além dos resultados, este projeto foi importante para praticar um fluxo completo de análise: leitura e preparação de uma base grande, estatística descritiva, identificação de outliers, visualização e interpretação dos resultados.

Como limitação, os dados mostram detecções feitas por satélite, mas não informam diretamente a causa de cada foco. Uma continuação natural seria aprofundar a dimensão espacial e temporal dos registros.
